# ==============================================================================

# MILESTONE 0: EXPLORATORY DATA ANALYSIS (EDA) & DATA PREPARATION

# ==============================================================================

# 1. Install and Import Required Libraries

!pip install datasets transformers matplotlib seaborn openpyxl

import os import numpy as np import pandas as pd import
matplotlib.pyplot as plt import seaborn as sns from datasets import
load_dataset

# Set plotting style for academic presentation

sns.set_theme(style=“whitegrid”) plt.rcParams\[“figure.figsize”\] = (10,
6)

# ——————————————————————————

# 2. Load the Dataset Configuration (Cell Phones and Accessories)

# ——————————————————————————

print(“Status: Loading product reviews from Hugging Face…”)
reviews_dataset = load_dataset( “McAuley-Lab/Amazon-Reviews-2023”,
“raw_review_Cell_Phones_and_Accessories”, trust_remote_code=True )
df_reviews = pd.DataFrame(reviews_dataset\[‘train’\])

print(“Status: Loading product metadata from Hugging Face…”)
meta_dataset = load_dataset( “McAuley-Lab/Amazon-Reviews-2023”,
“raw_meta_Cell_Phones_and_Accessories”, split=“full”,
trust_remote_code=True ) df_meta = pd.DataFrame(meta_dataset)

print(f”Initialization Complete. Total Reviews: {len(df_reviews)}, Total
Products: {len(df_meta)}“)

# ——————————————————————————

# 3. Missing Data & Quality Report (Syllabus Requirement)

# ——————————————————————————

print(“— Generating Missing Data Report —”) def
generate_missing_data_report(df, name): missing_count =
df.isnull().sum() missing_percent = (df.isnull().sum() / len(df)) \* 100
report_df = pd.DataFrame({‘Missing Count’: missing_count, ‘Percentage
(%)’: missing_percent}) print(f”Data Breakdown for: {name}“)
print(report_df\[report_df\[‘Missing Count’\] \> 0\])

generate_missing_data_report(df_reviews, “Reviews Data”)
generate_missing_data_report(df_meta, “Metadata”)

# Clean rows with missing critical features

df_reviews = df_reviews.dropna(subset=\[‘rating’, ‘text’\]) df_meta =
df_meta.dropna(subset=\[‘parent_asin’\])

# ——————————————————————————

# 4. Data Merging & Feature Engineering

# ——————————————————————————

# Join the tables using product unique identifier (parent_asin)

df_merged = pd.merge(df_reviews, df_meta, on=‘parent_asin’, how=‘inner’)

# Calculate review lengths for distribution analysis

df_merged\[‘review_length’\] =
df_merged\[‘text’\].fillna(’’).apply(lambda x: len(x.split()))

# Calculate sample image availability metrics

def check_image_availability(images_column): \# Quantify data quality
issues regarding broken or missing image configurations valid_links =
images_column.apply(lambda x: 1 if isinstance(x, (list, np.ndarray)) and
len(x) \> 0 else 0) return valid_links.sum(), (valid_links.sum() /
len(images_column)) \* 100

available_images, availability_pct =
check_image_availability(df_merged\[‘images’\]) print(f”Asset Metrics:
Total rows with images: {available_images} ({availability_pct:.2f}%)“)

# ——————————————————————————

# 5. Exploratory Visualizations (Save as Assets)

# ——————————————————————————

# A. Distribution of Ratings (Quantifying Rating Imbalance)

plt.figure() sns.countplot(data=df_merged, x=‘rating’,
palette=‘viridis’) plt.title(“Distribution of Product Ratings (Target
Feature Imbalance Analysis)”) plt.xlabel(“Rating Value”)
plt.ylabel(“Count”) plt.savefig(“rating_distribution.png”, dpi=300)
plt.show()

# B. Distribution of Review Lengths

plt.figure() sns.histplot(data=df_merged, x=‘review_length’, bins=50,
kde=True, color=‘blue’) plt.xlim(0, 300) \# Filter out heavy extreme
outliers for visualization clarity plt.title(“Distribution of Review
Lengths (Word Count)”) plt.xlabel(“Number of Words”)
plt.ylabel(“Density”) plt.savefig(“review_length_distribution.png”,
dpi=300) plt.show()

# C. Distribution of Product Prices

# Clean price text or extract float numeric values if necessary

df_merged\[‘clean_price’\] = pd.to_numeric(df_merged\[‘price’\],
errors=‘coerce’).fillna(0) plt.figure() sns.boxplot(data=df_merged,
x=‘clean_price’, color=‘green’) plt.xlim(0, 150) plt.title(“Distribution
Profile of Product Prices”) plt.xlabel(“Price (\$)”)
plt.savefig(“price_distribution.png”, dpi=300) plt.show()

# ——————————————————————————

# 6. Product-Level Data Splitting (Strict Prevention of Data Leakage)

# ——————————————————————————

# Get unique product IDs

unique_product_ids = df_merged\[‘parent_asin’\].unique()

# Deterministic shuffling for academic reproducibility

np.random.seed(42) np.random.shuffle(unique_product_ids)

# 70% Training, 15% Validation, 15% Testing data split thresholds

train_slice = int(0.70 \* len(unique_product_ids)) val_slice = int(0.85
\* len(unique_product_ids))

train_ids = unique_product_ids\[:train_slice\] val_ids =
unique_product_ids\[train_slice:val_slice\] test_ids =
unique_product_ids\[val_slice:\]

# Filter main merged dataframe into distinct splits based on product grouping

train_split = df_merged\[df_merged\[‘parent_asin’\].isin(train_ids)\]
val_split = df_merged\[df_merged\[‘parent_asin’\].isin(val_ids)\]
test_split = df_merged\[df_merged\[‘parent_asin’\].isin(test_ids)\]

print(“— Verification of Data Split Isolation —”) print(f”Training Set
Records : {len(train_split)} (Products: {len(train_ids)})“)
print(f”Validation Set Records : {len(val_split)} (Products:
{len(val_ids)})“) print(f”Testing Set Records : {len(test_split)}
(Products: {len(test_ids)})“)

# Check for data leakage

leakage_check = set(train_ids).intersection(set(test_ids)) print(f”Data
Leakage Detection Check: Overlapping items found = {len(leakage_check)}
(Success: Must be 0)“)

print(“: Milestone 0 Data Engineering Phase Successfully Executed.”)